# 04 — Inferential Analysis

This notebook performs exploratory statistical comparisons on the synthetic teaching dataset. Chi-square tests assess associations between categorical variables and the simulated churn label. Mann–Whitney U tests compare selected numeric feature distributions between simulated churned and retained accounts.

**Caution:** P-values are unadjusted and the data/labels are synthetic. Statistical significance is not evidence of causality or real-world business impact.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data/raw/enterprise_saas_churn_synthetic.csv'
if not DATA.exists():
    import runpy
    runpy.run_path(str(ROOT / 'scripts/generate_synthetic_dataset.py'))
df = pd.read_csv(DATA)
rows = []
for column in ['customer_tier', 'industry', 'region']:
    table = pd.crosstab(df[column], df['churned'])
    statistic, p_value, _, _ = chi2_contingency(table)
    v = np.sqrt(statistic / (table.values.sum() * min(table.shape[0]-1, table.shape[1]-1)))
    rows.append({'feature': column, 'test': 'Chi-square independence', 'statistic': statistic, 'p_value': p_value, 'cramers_v': v})
for column in ['license_utilization_pct', 'dau_mau_ratio', 'csat_score', 'support_cases_last_90d', 'days_to_renewal']:
    churned = df.loc[df.churned == 1, column].dropna()
    retained = df.loc[df.churned == 0, column].dropna()
    statistic, p_value = mannwhitneyu(churned, retained, alternative='two-sided')
    rows.append({'feature': column, 'test': 'Mann-Whitney U', 'statistic': statistic, 'p_value': p_value, 'cramers_v': np.nan})
results = pd.DataFrame(rows)
out = ROOT / 'data/processed/inferential_tests.csv'
out.parent.mkdir(exist_ok=True, parents=True)
results.to_csv(out, index=False)
print(f'Rows: {len(df):,} | simulated churn prevalence: {df.churned.mean():.3%}')
print(f'Results saved to {out.relative_to(ROOT)}')
display(results)


Rows: 3,000 | simulated churn prevalence: 15.267%
Results saved to data/processed/inferential_tests.csv


,feature,test,statistic,p_value,cramers_v
0,customer_tier,Chi-square independence,10.374853,5.586365e-03,0.058807
1,industry,Chi-square independence,2.124705,8.316346e-01,0.026613
2,region,Chi-square independence,0.807815,8.475969e-01,0.016410
3,license_utilization_pct,Mann-Whitney U,188955.500000,1.792169e-117,NaN
4,dau_mau_ratio,Mann-Whitney U,267371.500000,5.632199e-76,NaN
5,csat_score,Mann-Whitney U,261899.000000,5.767391e-78,NaN
6,support_cases_last_90d,Mann-Whitney U,783257.500000,8.383283e-33,NaN
7,days_to_renewal,Mann-Whitney U,504312.000000,5.117758e-06,NaN


## Interpretation

The numeric indicators differ significantly between simulated churned and retained accounts in this generated sample. Customer tier also has a statistically detectable association, but its Cramér's V is small. Industry and region do not show statistically significant associations in this run. Because the target is simulated from related signals, these results validate the programmed data-generation pattern rather than establish real customer behaviour.